# LSTM Stock Market Forecasting

This notebook builds a daily closing-price forecasting workflow around a Long
Short-Term Memory (LSTM) network. Technical indicators provide the input
signals. Principal Component Analysis (PCA) can be enabled as an optional
preprocessing step through one configuration value.

The goal is a clear, reproducible experiment rather than a trading strategy.

## Workflow

```text
market data -> technical indicators -> chronological split
            -> training-only scaling -> optional PCA -> sequences -> LSTM
```

The validation period controls early stopping. The test period is used only
after training has finished.

In [ ]:
from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import requests
import ta

from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow import keras


TICKER = "AAPL"
START_DATE = "2015-01-01"
END_DATE = "2023-01-01"
SEQUENCE_LENGTH = 60

# Keep this as None for the standard LSTM input pipeline.
# Set an integer, for example 4, to enable PCA with that many components.
PCA_COMPONENTS = None

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
EPOCHS = 100
BATCH_SIZE = 64
SEED = 42


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


set_seed(SEED)

## 1. Market data

The notebook downloads daily OHLCV data with `yfinance` and keeps a local CSV
cache. The cache makes later runs independent of small upstream API changes.

In [ ]:
def download_market_data(ticker, start_date, end_date):
    """Download daily OHLCV data from Yahoo's chart endpoint."""
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    response = requests.get(
        url,
        params={
            "period1": int(pd.Timestamp(start_date, tz="UTC").timestamp()),
            "period2": int(pd.Timestamp(end_date, tz="UTC").timestamp()),
            "interval": "1d",
            "events": "history",
        },
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=60,
    )
    response.raise_for_status()

    chart = response.json()["chart"]
    if chart.get("error") or not chart.get("result"):
        raise RuntimeError(f"Yahoo Finance returned no data: {chart.get('error')}")

    result = chart["result"][0]
    quote = result["indicators"]["quote"][0]
    index = (
        pd.to_datetime(result["timestamp"], unit="s", utc=True)
        .tz_convert(None)
        .normalize()
    )
    frame = pd.DataFrame(
        {
            "Open": quote["open"],
            "High": quote["high"],
            "Low": quote["low"],
            "Close": quote["close"],
            "Volume": quote["volume"],
        },
        index=index,
    )
    frame.index.name = "Date"
    return frame.sort_index().dropna()


def load_market_data(ticker, start_date, end_date, cache_dir="data"):
    cache_dir = Path(cache_dir)
    cache_path = cache_dir / f"{ticker.lower()}_{start_date}_{end_date}.csv"
    required = ["Open", "High", "Low", "Close", "Volume"]

    frame = None
    if cache_path.exists():
        try:
            cached = pd.read_csv(cache_path, index_col="Date", parse_dates=["Date"])
            if not cached.empty and set(required).issubset(cached.columns):
                frame = cached
        except (OSError, ValueError):
            pass

    if frame is None:
        frame = download_market_data(ticker, start_date, end_date)
        cache_dir.mkdir(parents=True, exist_ok=True)
        frame.to_csv(cache_path)

    missing = set(required) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing market-data columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError("No market data is available for the selected ticker and dates.")
    return frame[required].sort_index().dropna().copy()


market_data = load_market_data(TICKER, START_DATE, END_DATE)
market_data.tail()


## 2. Technical indicators

The feature set combines price, trend, momentum, volatility, and trend-strength
signals. Rolling indicators introduce missing rows at the beginning, which are
removed after all features have been calculated.

In [ ]:
FEATURE_COLUMNS = [
    "Close",
    "MA50",
    "SMA5",
    "RSI",
    "MACD",
    "BollingerUpper",
    "BollingerLower",
    "ATR",
    "ADX",
    "STD10",
]


def add_technical_indicators(frame):
    result = frame.copy()
    close = result["Close"]
    high = result["High"]
    low = result["Low"]

    result["MA50"] = close.rolling(50).mean()
    result["SMA5"] = close.rolling(5).mean()
    result["STD10"] = close.rolling(10).std()
    result["MACD"] = (
        close.ewm(span=12, adjust=False).mean()
        - close.ewm(span=26, adjust=False).mean()
    )

    ma10 = close.rolling(10).mean()
    result["BollingerUpper"] = ma10 + 2 * result["STD10"]
    result["BollingerLower"] = ma10 - 2 * result["STD10"]

    change = close.diff()
    average_gain = change.clip(lower=0).rolling(14).mean()
    average_loss = -change.clip(upper=0).rolling(14).mean()
    relative_strength = average_gain / average_loss.replace(0, np.nan)
    result["RSI"] = 100 - 100 / (1 + relative_strength)

    true_range = pd.concat(
        [
            high - low,
            (high - close.shift()).abs(),
            (low - close.shift()).abs(),
        ],
        axis=1,
    ).max(axis=1)
    result["ATR"] = true_range.rolling(14).mean()
    result["ADX"] = ta.trend.ADXIndicator(
        high=high,
        low=low,
        close=close,
        window=14,
    ).adx()

    return result.dropna().copy()


model_data = add_technical_indicators(market_data)
print(f"Rows available for modelling: {len(model_data):,}")
model_data[FEATURE_COLUMNS].tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_data.index, model_data["Close"], label="Close", linewidth=1.3)
ax.plot(model_data.index, model_data["MA50"], label="50-day moving average", linewidth=1)
ax.set(title=f"{TICKER} closing price", xlabel="Date", ylabel="Price")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

## 3. Chronological preprocessing with optional PCA

Scaling and PCA are fitted only on training rows. All sequences are assigned to
a split according to the date of their prediction target. Validation and test
windows may use earlier observations, but never future ones.

In [ ]:
def prepare_datasets(
    frame,
    feature_columns,
    sequence_length,
    pca_components=None,
    train_fraction=0.70,
    validation_fraction=0.15,
):
    features = frame[feature_columns].to_numpy(dtype=np.float64)
    targets = frame[["Close"]].to_numpy(dtype=np.float64)
    dates = frame.index.to_numpy()

    train_end = int(len(frame) * train_fraction)
    validation_end = int(len(frame) * (train_fraction + validation_fraction))
    if sequence_length >= train_end:
        raise ValueError("The training period must be longer than the sequence length.")

    feature_scaler = StandardScaler()
    feature_scaler.fit(features[:train_end])
    transformed_features = feature_scaler.transform(features)

    pca = None
    if pca_components is not None:
        maximum = min(transformed_features[:train_end].shape)
        if not 1 <= pca_components <= maximum:
            raise ValueError(f"PCA components must be between 1 and {maximum}.")
        pca = PCA(n_components=pca_components)
        pca.fit(transformed_features[:train_end])
        transformed_features = pca.transform(transformed_features)

    target_scaler = StandardScaler()
    target_scaler.fit(targets[:train_end])
    transformed_targets = target_scaler.transform(targets).reshape(-1)

    buckets = {
        name: {"X": [], "y": [], "dates": [], "last_close": []}
        for name in ("train", "validation", "test")
    }

    for target_index in range(sequence_length, len(frame)):
        if target_index < train_end:
            split = "train"
        elif target_index < validation_end:
            split = "validation"
        else:
            split = "test"

        buckets[split]["X"].append(
            transformed_features[target_index - sequence_length : target_index]
        )
        buckets[split]["y"].append(transformed_targets[target_index])
        buckets[split]["dates"].append(dates[target_index])
        buckets[split]["last_close"].append(targets[target_index - 1, 0])

    datasets = {}
    for name, values in buckets.items():
        datasets[name] = {
            "X": np.asarray(values["X"], dtype=np.float32),
            "y": np.asarray(values["y"], dtype=np.float32),
            "dates": np.asarray(values["dates"]),
            "last_close": np.asarray(values["last_close"]),
        }

    transformers = {
        "feature_scaler": feature_scaler,
        "target_scaler": target_scaler,
        "pca": pca,
    }
    return datasets, transformers

In [ ]:
datasets, transformers = prepare_datasets(
    model_data,
    FEATURE_COLUMNS,
    sequence_length=SEQUENCE_LENGTH,
    pca_components=PCA_COMPONENTS,
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
)

mode = "standardized indicators" if PCA_COMPONENTS is None else f"{PCA_COMPONENTS} PCA components"
print(f"Input mode: {mode}")
for split_name, split_data in datasets.items():
    print(f"{split_name:>10}: X={split_data['X'].shape}, y={split_data['y'].shape}")

In [ ]:
pca = transformers["pca"]
if pca is not None:
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
    plt.figure(figsize=(8, 4))
    plt.bar(range(1, len(cumulative_variance) + 1), pca.explained_variance_ratio_)
    plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
    plt.xlabel("Principal component")
    plt.ylabel("Explained variance ratio")
    plt.title("PCA explained variance fitted on training data")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
else:
    print("PCA is disabled. Set PCA_COMPONENTS to an integer to enable it.")

## 4. LSTM model

The model consumes the prepared sequences regardless of whether PCA is active.
Early stopping uses the validation period, and sequence shuffling remains off
to preserve time order.

In [ ]:
def build_lstm(input_shape):
    inputs = keras.Input(shape=input_shape, name="feature_sequence")
    x = keras.layers.LSTM(64, return_sequences=True)(inputs)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.LSTM(32)(x)
    x = keras.layers.Dropout(0.2)(x)
    x = keras.layers.Dense(16, activation="relu")(x)
    outputs = keras.layers.Dense(1, name="scaled_close")(x)

    model = keras.Model(inputs, outputs, name="stock_market_lstm")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),
        loss=keras.losses.Huber(),
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


model = build_lstm(datasets["train"]["X"].shape[1:])
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        min_delta=1e-4,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=7,
        min_lr=1e-6,
    ),
]

history = model.fit(
    datasets["train"]["X"],
    datasets["train"]["y"],
    validation_data=(datasets["validation"]["X"], datasets["validation"]["y"]),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=False,
    verbose=1,
)

## 5. Test-period evaluation

The LSTM is evaluated against a simple persistence baseline that predicts the
next close will equal the latest observed close. A forecasting model should be
interpreted in relation to this baseline rather than from error values alone.

In [ ]:
target_scaler = transformers["target_scaler"]
predicted_scaled = model.predict(datasets["test"]["X"], verbose=0)
predicted_close = target_scaler.inverse_transform(predicted_scaled).reshape(-1)
actual_close = target_scaler.inverse_transform(
    datasets["test"]["y"].reshape(-1, 1)
).reshape(-1)
baseline_close = datasets["test"]["last_close"]


def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "MAPE (%)": mean_absolute_percentage_error(actual, predicted) * 100,
    }


metrics = pd.DataFrame(
    {
        "LSTM": regression_metrics(actual_close, predicted_close),
        "Last-close baseline": regression_metrics(actual_close, baseline_close),
    }
).T
metrics.round(3)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(history.history["loss"], label="Training loss")
axes[0].plot(history.history["val_loss"], label="Validation loss")
axes[0].set(title="Training history", xlabel="Epoch", ylabel="Huber loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(datasets["test"]["dates"], actual_close, label="Actual", linewidth=1.5)
axes[1].plot(datasets["test"]["dates"], predicted_close, label="LSTM", linewidth=1.2)
axes[1].plot(
    datasets["test"]["dates"],
    baseline_close,
    label="Last-close baseline",
    linewidth=1,
    alpha=0.75,
)
axes[1].set(title="Test-period forecasts", xlabel="Date", ylabel="Closing price")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()

## Interpretation and limitations

- PCA is optional and should be treated as a modelling choice, not an assumed
  improvement.
- Results depend on the selected asset, time period, feature set, and random
  seed.
- The experiment uses one chronological holdout rather than walk-forward
  validation.
- Transaction costs, slippage, and trading decisions are outside the scope of
  this notebook.

This project is for educational experimentation and is not financial advice.